# Chapter 5. pandas Fundamentals

*Data Analytics with Python and Jupyter Notebook*

Companion notebook. Run each cell in order with **Shift + Enter**, and read the chapter alongside it.

## 5.1 What pandas adds to NumPy

As always, start by importing the libraries:

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Keep printed tables narrow enough for the book's pages
pd.set_option("display.width", 80)

## 5.2 Series: one labeled column

### Creating a Series

You can make a Series from a list:

In [3]:
prices = pd.Series([3.00, 4.50, 4.25, 4.75, 3.25, 2.95])
prices

0    3.00
1    4.50
2    4.25
3    4.75
4    3.25
5    2.95
dtype: float64

Numbers are not very good labels for prices. Pass `index=` to supply your own:

In [4]:
products = ["Espresso", "Latte", "Cappuccino",
            "Cold Brew", "Croissant", "Blueberry Muffin"]
prices = pd.Series([3.00, 4.50, 4.25, 4.75, 3.25, 2.95],
                   index=products, name="unit_price")
prices

Espresso            3.00
Latte               4.50
Cappuccino          4.25
Cold Brew           4.75
Croissant           3.25
Blueberry Muffin    2.95
Name: unit_price, dtype: float64

Now each price is labeled with its product, and the Series has a **name**, `unit_price`, which becomes its column name if you later put it in a table. A Python dictionary (Chapter 3) works too: the keys become the index.

In [5]:
pd.Series({"Downtown": 1.4, "University": 1.1, "Riverside": 0.8})

Downtown      1.4
University    1.1
Riverside     0.8
dtype: float64

### Looking things up by label

With labels, you can ask for a value by name:

In [6]:
prices["Latte"]

np.float64(4.5)

And all the NumPy tricks from Chapter 4 still work. Vectorized math applies to every value, and a boolean mask keeps the matching ones:

In [7]:
prices[prices > 4]

Latte         4.50
Cappuccino    4.25
Cold Brew     4.75
Name: unit_price, dtype: float64

The two parts of a Series are available separately as `.index` and `.values`:

In [8]:
print(prices.index)
print(prices.to_numpy())

Index(['Espresso', 'Latte', 'Cappuccino', 'Cold Brew', 'Croissant',
       'Blueberry Muffin'],
      dtype='str')
[3.   4.5  4.25 4.75 3.25 2.95]


### Math lines up by label

Here is the big difference from NumPy. When you combine two Series, pandas matches values by **label**, not by position. Suppose Bean There counts croissants and lattes from two different systems, and the second system lists the products in a different order:

In [9]:
monday = pd.Series({"Latte": 41, "Croissant": 22, "Espresso": 19})
tuesday = pd.Series({"Espresso": 17, "Latte": 45, "Scone": 6})
monday + tuesday

Croissant     NaN
Espresso     36.0
Latte        86.0
Scone         NaN
dtype: float64

This **alignment** is usually exactly what you want, and it prevents a whole class of mistakes where two lists are quietly out of order. If you would rather treat a missing value as zero, use the `.add()` method with `fill_value`:

In [10]:
monday.add(tuesday, fill_value=0)

Croissant    22.0
Espresso     36.0
Latte        86.0
Scone         6.0
dtype: float64

## 5.3 DataFrames: a table of Series

### Creating a DataFrame

You can build a small DataFrame from a dictionary of lists. Each key becomes a column name:

In [11]:
stores = pd.DataFrame({
    "store": ["Downtown", "University", "Riverside"],
    "opened": [2019, 2021, 2022],
    "seats": [40, 55, 24],
    "manager": ["Priya Nair", "Tom Okafor", "Lena Fischer"],
})
stores

,store,opened,seats,manager
0,Downtown,2019,40,Priya Nair
1,University,2021,55,Tom Okafor
2,Riverside,2022,24,Lena Fischer


Most of the time, though, you load a DataFrame from a file. Here is the Bean There sales file again:

In [12]:
sales = pd.read_csv("../data/bean_there_sales.csv", parse_dates=["date"])
sales.head()

,date,store,product,category,quantity,unit_price,revenue
0,2026-01-01,Downtown,Espresso,Coffee,30,3.00,90.00
1,2026-01-01,Downtown,Latte,Coffee,53,4.50,238.50
2,2026-01-01,Downtown,Cappuccino,Coffee,35,4.25,148.75
3,2026-01-01,Downtown,Cold Brew,Coffee,14,4.75,66.50
4,2026-01-01,Downtown,Croissant,Bakery,26,3.25,84.50


### Anatomy of a DataFrame

- The **index**: the row labels down the left side (here 0, 1, 2, ...).
- The **columns**: the column labels across the top.
- The **values**: the data itself, stored column by column.

In [13]:
print(sales.shape)
print(sales.columns)
print(sales.index)

(1620, 7)
Index(['date', 'store', 'product', 'category', 'quantity', 'unit_price',
       'revenue'],
      dtype='str')
RangeIndex(start=0, stop=1620, step=1)


`.dtypes` lists the type of every column:

In [14]:
sales.dtypes

date          datetime64[us]
store                    str
product                  str
category                 str
quantity               int64
unit_price           float64
revenue              float64
dtype: object

### First looks

You met `.head()`, `.info()` and `.describe()` in Chapter 1. Three more are worth knowing. `.tail()` shows the last rows, and `.sample()` picks random rows, which is a better way to spot problems than always looking at the top of a file:

In [15]:
sales.tail(3)

,date,store,product,category,quantity,unit_price,revenue
1617,2026-03-31,Riverside,Cold Brew,Coffee,17,4.75,80.75
1618,2026-03-31,Riverside,Croissant,Bakery,25,3.25,81.25
1619,2026-03-31,Riverside,Blueberry Muffin,Bakery,13,2.95,38.35


This table is too wide for the page, so pandas wrapped it: the backslash `\` at the end of the header line means "continued below", and the `revenue` column appears underneath. In Jupyter you will see the full table in one piece, with a scroll bar if needed.

In [16]:
sales.sample(3, random_state=1)

,date,store,product,category,quantity,unit_price,revenue
1162,2026-03-06,University,Croissant,Bakery,21,3.25,68.25
1296,2026-03-14,Downtown,Espresso,Coffee,26,3.00,78.00
301,2026-01-17,Riverside,Latte,Coffee,37,4.50,166.50


For a single column, `.unique()`, `.nunique()` and `.value_counts()` tell you what values it contains:

In [17]:
print(sales["store"].unique().tolist())
print(sales["product"].nunique())

['Downtown', 'University', 'Riverside']
6


In [18]:
sales["category"].value_counts()

category
Coffee    1080
Bakery     540
Name: count, dtype: int64

## 5.4 Selecting columns

### One column gives a Series

Square brackets with a single column name return that column as a Series:

In [19]:
qty = sales["quantity"]
print(type(qty))
qty.head()

<class 'pandas.Series'>


0    30
1    53
2    35
3    14
4    26
Name: quantity, dtype: int64

### A list of columns gives a DataFrame

To select several columns, put a **list** of names inside the brackets. That means two pairs of square brackets: the outer pair does the selecting, the inner pair makes the list.

In [20]:
sales[["date", "store", "revenue"]].head()

,date,store,revenue
0,2026-01-01,Downtown,90.00
1,2026-01-01,Downtown,238.50
2,2026-01-01,Downtown,148.75
3,2026-01-01,Downtown,66.50
4,2026-01-01,Downtown,84.50


### Dot notation

## 5.5 Selecting rows with `loc` and `iloc`

### `iloc`: by position

`iloc` works exactly like NumPy indexing from Chapter 4: `[rows, columns]`, zero-based, with slices that stop **before** the end:

In [21]:
sales.iloc[0]

date          2026-01-01 00:00:00
store                    Downtown
product                  Espresso
category                   Coffee
quantity                       30
unit_price                    3.0
revenue                      90.0
Name: 0, dtype: object

A single position returns one row as a Series, with the column names as its index. Slices and lists work as usual:

In [22]:
sales.iloc[:3, :4]

,date,store,product,category
0,2026-01-01,Downtown,Espresso,Coffee
1,2026-01-01,Downtown,Latte,Coffee
2,2026-01-01,Downtown,Cappuccino,Coffee


In [23]:
sales.iloc[[0, 500, 1000], [0, 1, 6]]

,date,store,revenue
0,2026-01-01,Downtown,90.0
500,2026-01-28,Riverside,68.0
1000,2026-02-25,University,71.5


### `loc`: by label

`loc` uses labels. With the default `RangeIndex`, the row labels happen to be the same numbers as the positions, which makes `loc` and `iloc` look alike. They are not. Compare these two:

In [24]:
sales.loc[0:2, ["store", "product"]]

,store,product
0,Downtown,Espresso
1,Downtown,Latte
2,Downtown,Cappuccino


In [25]:
sales.iloc[0:2, [1, 2]]

,store,product
0,Downtown,Espresso
1,Downtown,Latte


### A meaningful index

`loc` really shines once the index holds meaningful labels. Our `stores` table is a good example: the store name identifies each row, so make it the index with `.set_index()`:

In [26]:
stores = stores.set_index("store")
stores

,opened,seats,manager
store,,,
Downtown,2019,40,Priya Nair
University,2021,55,Tom Okafor
Riverside,2022,24,Lena Fischer


The `store` column moved to the left and became the row labels. Now you can look up rows by name:

In [27]:
stores.loc["University"]

opened           2021
seats              55
manager    Tom Okafor
Name: University, dtype: object

In [28]:
stores.loc["University", "manager"]

'Tom Okafor'

`.set_index()` returns a new DataFrame, so we assigned the result back to `stores`. To turn the index back into an ordinary column, use `.reset_index()`:

In [29]:
stores.reset_index()

,store,opened,seats,manager
0,Downtown,2019,40,Priya Nair
1,University,2021,55,Tom Okafor
2,Riverside,2022,24,Lena Fischer


### Dates in the index

The index can hold dates too. Let's take Downtown's Latte sales and index them by date:

In [30]:
dt_latte = (
    sales[(sales["store"] == "Downtown") & (sales["product"] == "Latte")]
    .set_index("date")
)
dt_latte[["quantity", "revenue"]].head(3)

,quantity,revenue
date,,
2026-01-01,53,238.5
2026-01-02,52,234.0
2026-01-03,64,288.0


(The first line is a filter, which Section 5.6 explains in full.) Now `loc` accepts dates written as text, and a date slice includes both ends:

In [31]:
dt_latte.loc["2026-01-24", "quantity"]

np.int64(76)

In [32]:
dt_latte.loc["2026-03-28":"2026-03-31", ["quantity", "revenue"]]

,quantity,revenue
date,,
2026-03-28,74,333.0
2026-03-29,62,279.0
2026-03-30,39,175.5
2026-03-31,52,234.0


Remember the busiest Latte day from Chapter 4, which NumPy could only report as "position 23"? With a date index, you can ask for the date directly using `.idxmax()`, which returns the **label** of the largest value:

In [33]:
dt_latte["quantity"].idxmax()

Timestamp('2026-03-21 00:00:00')

## 5.6 Filtering rows

### One condition

Compare a column with a value and you get a Series of `True` and `False`, one per row:

In [34]:
is_cold_brew = sales["product"] == "Cold Brew"
is_cold_brew.head()

0    False
1    False
2    False
3     True
4    False
Name: product, dtype: bool

Put that mask in square brackets and pandas keeps the `True` rows:

In [35]:
cold_brew = sales[is_cold_brew]
print(cold_brew.shape)
cold_brew.head(3)

(270, 7)


,date,store,product,category,quantity,unit_price,revenue
3,2026-01-01,Downtown,Cold Brew,Coffee,14,4.75,66.50
9,2026-01-01,University,Cold Brew,Coffee,19,4.75,90.25
15,2026-01-01,Riverside,Cold Brew,Coffee,7,4.75,33.25


Usually you write the condition directly inside the brackets:

In [36]:
sales[sales["quantity"] >= 70]

,date,store,product,category,quantity,unit_price,revenue
415,2026-01-24,Downtown,Latte,Coffee,76,4.5,342.0
667,2026-02-07,Downtown,Latte,Coffee,70,4.5,315.0
811,2026-02-15,Downtown,Latte,Coffee,70,4.5,315.0
1423,2026-03-21,Downtown,Latte,Coffee,79,4.5,355.5
1549,2026-03-28,Downtown,Latte,Coffee,74,4.5,333.0


### Several conditions

Combine conditions with `&` (and), `|` (or) and `~` (not), wrapping each one in parentheses, exactly as with NumPy arrays:

In [37]:
weekend_rb = sales[
    (sales["store"] == "Riverside")
    & (sales["category"] == "Bakery")
    & (sales["date"].dt.dayofweek >= 5)
]
weekend_rb[["date", "store", "product", "quantity"]].head(4)

,date,store,product,quantity
52,2026-01-03,Riverside,Croissant,21
53,2026-01-03,Riverside,Blueberry Muffin,8
70,2026-01-04,Riverside,Croissant,21
71,2026-01-04,Riverside,Blueberry Muffin,12


### Handy shortcuts: `isin`, `between`, and `str`

`.isin()` tests whether each value is in a list, which replaces a long chain of `|` conditions:

In [38]:
bakery_items = ["Croissant", "Blueberry Muffin"]
sales[sales["product"].isin(bakery_items)].shape

(540, 7)

`.between()` tests whether values fall in a range, including both ends:

In [39]:
feb = sales[sales["date"].between("2026-02-01", "2026-02-28")]
print(feb["date"].min(), "to", feb["date"].max())
print(len(feb), "rows")

2026-02-01 00:00:00 to 2026-02-28 00:00:00
504 rows


Text columns have a `.str` accessor with string methods that work on every value at once. For example, `.str.startswith()` checks how each value begins. Here we use it inside `.loc`, with the mask for the rows and `"product"` for the column:

In [40]:
c_rows = sales.loc[sales["product"].str.startswith("C"), "product"]
c_rows.unique().tolist()

['Cappuccino', 'Cold Brew', 'Croissant']

### `query`: filtering with a sentence

`.query()` lets you write the condition as a string, which some people find easier to read. Inside the string, you refer to columns by their bare names, and you can use `and`, `or` and `not`:

In [41]:
sales.query("store == 'University' and quantity > 50")

,date,store,product,category,quantity,unit_price,revenue
763,2026-02-12,University,Latte,Coffee,52,4.5,234.0
997,2026-02-25,University,Latte,Coffee,54,4.5,243.0
1249,2026-03-11,University,Latte,Coffee,52,4.5,234.0


## 5.7 Sorting

### Sorting by values

`.sort_values()` sorts rows by one or more columns. By default it sorts from smallest to largest:

In [42]:
cols = ["date", "store", "product", "quantity", "revenue"]
sales.sort_values("revenue")[cols].head(3)

,date,store,product,quantity,revenue
173,2026-01-10,University,Blueberry Muffin,5,14.75
33,2026-01-02,Riverside,Cold Brew,4,19.00
233,2026-01-13,Riverside,Blueberry Muffin,7,20.65


Two of the three smallest sales are Blueberry Muffins, the cheapest product, and two are at Riverside, the quietest store. To save space we stored the column list in `cols` and selected those columns after sorting. Add `ascending=False` to sort from largest to smallest:

In [43]:
sales.sort_values("revenue", ascending=False)[cols].head(3)

,date,store,product,quantity,revenue
1423,2026-03-21,Downtown,Latte,79,355.5
415,2026-01-24,Downtown,Latte,76,342.0
1549,2026-03-28,Downtown,Latte,74,333.0


### Sorting by several columns

Pass a list of columns to sort by the first, then break ties with the next. You can also give a list for `ascending`, one entry per column. Here we sort by store alphabetically, and within each store by quantity from highest to lowest:

In [44]:
(
    sales[sales["date"] == "2026-03-31"]
    .sort_values(["store", "quantity"], ascending=[True, False])
    .loc[:, ["store", "product", "quantity"]]
    .head(7)
)

,store,product,quantity
1603,Downtown,Latte,52
1604,Downtown,Cappuccino,38
1602,Downtown,Espresso,29
1606,Downtown,Croissant,29
1605,Downtown,Cold Brew,28
1607,Downtown,Blueberry Muffin,22
1618,Riverside,Croissant,25


### `nlargest` and `nsmallest`

"Top 5" questions are so common that pandas has shortcuts. `.nlargest(n, column)` is equivalent to sorting in descending order and taking the head, but quicker to type:

In [45]:
sales.nlargest(5, "quantity")[["date", "store", "product", "quantity"]]

,date,store,product,quantity
1423,2026-03-21,Downtown,Latte,79
415,2026-01-24,Downtown,Latte,76
1549,2026-03-28,Downtown,Latte,74
667,2026-02-07,Downtown,Latte,70
811,2026-02-15,Downtown,Latte,70


### Sorting by the index

`.sort_index()` sorts by the row labels instead. It is useful after a filter or a sort has shuffled them, and it works on a Series too:

In [46]:
prices.sort_index()

Blueberry Muffin    2.95
Cappuccino          4.25
Cold Brew           4.75
Croissant           3.25
Espresso            3.00
Latte               4.50
Name: unit_price, dtype: float64

## 5.8 Creating and changing columns

### Arithmetic on columns

To create a new column, assign to a new name in square brackets. Arithmetic between columns is vectorized, just like NumPy:

In [47]:
sales["unit_cost"] = sales["unit_price"] * 0.35
sales["profit"] = sales["revenue"] - sales["quantity"] * sales["unit_cost"]
sales[["product", "quantity", "unit_price", "unit_cost", "profit"]].head(3)

,product,quantity,unit_price,unit_cost,profit
0,Espresso,30,3.00,1.0500,58.5000
1,Latte,53,4.50,1.5750,155.0250
2,Cappuccino,35,4.25,1.4875,96.6875


To change an existing column, assign to its name. Let's round `unit_cost` to whole cents and recompute `profit`:

In [48]:
sales["unit_cost"] = sales["unit_cost"].round(2)
sales["profit"] = sales["revenue"] - sales["quantity"] * sales["unit_cost"]
print(f"Total profit: ${sales['profit'].sum():,.2f}")

Total profit: $104,533.93


### Columns from conditions with `np.where`

`np.where` from Chapter 4 creates a column with one of two values depending on a condition. Let's mark weekends:

In [49]:
is_weekend = sales["date"].dt.dayofweek >= 5
sales["day_type"] = np.where(is_weekend, "Weekend", "Weekday")
sales["day_type"].value_counts()

day_type
Weekday    1152
Weekend     468
Name: count, dtype: int64

### Mapping values with a dictionary

`.map()` replaces each value using a dictionary. Say Bean There wants to label each store's neighborhood type:

In [50]:
area = {"Downtown": "Business", "University": "Campus",
        "Riverside": "Residential"}
sales["area"] = sales["store"].map(area)
sales[["store", "area"]].drop_duplicates()

,store,area
0,Downtown,Business
6,University,Campus
12,Riverside,Residential


### `assign`: creating columns in a chain

The `sales["new"] = ...` style changes `sales` in place. `.assign()` does the same job but returns a new DataFrame, so it fits inside a method chain:

In [51]:
(
    sales
    .assign(avg_price=sales["revenue"] / sales["quantity"])
    .loc[:, ["product", "revenue", "quantity", "avg_price"]]
    .head(3)
)

,product,revenue,quantity,avg_price
0,Espresso,90.00,30,3.00
1,Latte,238.50,53,4.50
2,Cappuccino,148.75,35,4.25


### Renaming, dropping, and changing types

- `.rename(columns={"old": "new"})` renames columns.
- `.drop(columns=[...])` removes columns.
- `.astype(type)` converts a column to a different dtype.

In [52]:
tidy = (
    sales
    .drop(columns=["unit_cost", "area"])
    .rename(columns={"unit_price": "price", "day_type": "day"})
)
print(", ".join(tidy.columns))

date, store, product, category, quantity, price, revenue, profit, day


`.astype()` is most useful for text that should be numbers, a problem Chapter 7 tackles. Here we convert `quantity` to decimals, just to see it work, without saving the result:

In [53]:
print(sales["quantity"].astype("float64").head(3).tolist())
print(sales["quantity"].dtype)

[30.0, 53.0, 35.0]
int64


### Changing some rows with `loc`

Often you want to change values only in rows that meet a condition. Suppose Bean There ran a Cold Brew promotion at Riverside in March at $4.25 instead of $4.75, and the till recorded the wrong price. Use `.loc` with a mask for the rows and a column name, all in **one** step:

In [54]:
promo = (
    (sales["store"] == "Riverside")
    & (sales["product"] == "Cold Brew")
    & (sales["date"].dt.month == 3)
)
sales.loc[promo, "unit_price"] = 4.25
sales.loc[promo, "revenue"] = sales.loc[promo, "quantity"] * 4.25
sales.loc[promo, ["date", "store", "unit_price", "revenue"]].head(3)

,date,store,unit_price,revenue
1077,2026-03-01,Riverside,4.25,72.25
1095,2026-03-02,Riverside,4.25,85.00
1113,2026-03-03,Riverside,4.25,68.00


### Copy-on-write: why the "one step" matters

That rule has one important consequence. The following looks like it should change prices in `sales`, but it does not:

*Example to try on your own machine (not run here):*

```python
# Don't do this: it changes a temporary copy, not sales
sales[sales["store"] == "Riverside"]["unit_price"] = 9.99
```

The flip side is reassuring. When you filter a subset and change it, your original data is safe:

In [55]:
riverside = sales[sales["store"] == "Riverside"]
riverside["unit_price"] = 0
print(riverside["unit_price"].max())
print(sales["unit_price"].max())

0
4.75


## 5.9 Putting it together

First, reload a clean copy of the data, so the promotion and the zeroed prices from the last section do not affect the answer:

In [56]:
sales = pd.read_csv("../data/bean_there_sales.csv", parse_dates=["date"])

Then filter, create columns, select, and sort in one readable chain:

In [57]:
weekend = (
    sales[sales["date"].dt.dayofweek >= 5]
    .assign(profit=lambda d: d["revenue"] * 0.65)
    .loc[:, ["date", "store", "product", "revenue", "profit"]]
)
weekend.head(3)

,date,store,product,revenue,profit
36,2026-01-03,Downtown,Espresso,102.00,66.3000
37,2026-01-03,Downtown,Latte,288.00,187.2000
38,2026-01-03,Downtown,Cappuccino,182.75,118.7875


Now, which product earned the most at each store on a single weekend day? Sort by revenue from highest to lowest and keep the first row for each store with `.drop_duplicates()`:

In [58]:
(
    weekend
    .sort_values("revenue", ascending=False)
    .drop_duplicates(subset="store")
)

,date,store,product,revenue,profit
1423,2026-03-21,Downtown,Latte,355.5,231.075
1183,2026-03-07,Riverside,Latte,216.0,140.400
1447,2026-03-22,University,Latte,193.5,125.775


## 5.11 Exercises

Open `ch05_pandas_fundamentals.ipynb` and add your answers at the bottom. Solutions are in Appendix D.

1. Create a Series of the three stores' seat counts (Downtown 40, University 55, Riverside 24) indexed by store name. Which store has the most seats? *Hint:* `.idxmax()`.
2. Select the `date`, `product` and `quantity` columns for rows 100 to 104 of `sales`, first with `iloc` and then with `loc`. What do you need to change in the slice to get the same five rows?
3. How many rows record a University sale of more than 40 units of anything? Which products are they? *Hint:* filter, then `.value_counts()` on `product`.
4. Find the five lowest-revenue Latte rows across all stores. Which store do they come from?
5. Filter the sales to Espresso and Cappuccino at Downtown in January using `.isin()` and `.between()`. How many rows do you get, and is that the number you expect?
6. Add a column `price_band` that is "Premium" when `unit_price` is at least 4.50 and "Standard" otherwise. How many rows fall in each band?
7. Bean There has decided the Blueberry Muffin price should have been $3.10 for all of March. Update `unit_price` and `revenue` for those rows with `.loc`, then check the new total March muffin revenue. *Hint:* reload the data first so earlier changes don't interfere.
8. **Think about it:** run `sales[sales["store"] == "Downtown"]["quantity"] = 0` in a new cell. What message do you see, and did `sales` change? Rewrite it so that it would really change the data (then reload the data afterwards).